In [ ]:
import openslide as ops
from glob import glob
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import cv2
import anndata as ad
import pandas as pd
import h5py
import json
from tqdm import tqdm
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:

class_list = {
    0: "Epithelial",
    1: "Stromal",
    2: "Lymphoplasmacytic",
    3: "Granulocyte",
}

class_colors_hex = {
    "Epithelial": "#FF0000",           # 빨강
    "Stromal": "#00FF00",              # 초록
    "Lymphoplasmacytic": "#FFFF00",    # 노랑
    "Granulocyte": "#1E90FF",          # DodgerBlue (밝은 파랑)       # 회색
}

class_colors = {
    "Epithelial": [255, 0, 0],         # 빨강
    "Stromal": [0, 255, 0],            # 초록
    "Lymphoplasmacytic": [255, 255, 0], # 노랑
    "Granulocyte": [30, 144, 255],     # 도저블루 (밝은 파랑)
}

class_list_inv = {v: k for k, v in class_list.items()}

In [ ]:
annotation_list=glob('../../data/spatialTranscriptome/preprocessed_xenium/labels/TEN*.csv')
wsi_list=[f.replace('/labels','/wsis').replace('.csv','.tif') for f in annotation_list]
metadata_list=[f.replace('/wsis','/metadata').replace('.tif','.json') for f in wsi_list]
coord_list=[f.replace('/wsis','/patches').replace('.tif','.h5') for f in wsi_list]

In [ ]:

x20_mpp=0.5
x5_mpp=2.0
patch_image_size=512
for i in range(18,len(wsi_list)):
    slide=ops.OpenSlide(wsi_list[i])
    width, height = slide.dimensions
    metadata_file=metadata_list[i]
    annotation_file=annotation_list[i]
    coord_file=coord_list[i]
    with h5py.File(coord_file, "r") as f:
        coords = f["coords"][:]
    with open(metadata_file, 'r') as f:
        metadata = f.read()
    metadata = json.loads(metadata)
    annotation_df=pd.read_csv(annotation_file)
    x_min=coords[:,0].min()+1000
    y_min=coords[:,1].min()+1000
    x_max=coords[:,0].max()-1000
    y_max=coords[:,1].max()-1000
    # 우선순위 2: pixel_size_um_estimated
    if 'pixel_size_um_estimated' in metadata and metadata['pixel_size_um_estimated'] is not None:
        slide_mpp = metadata['pixel_size_um_estimated']
        source = "pixel_size_um_estimated"
    
    # 우선순위 3: pixel_size (Xenium 장비의 native pixel size)
    elif 'pixel_size' in metadata and metadata['pixel_size'] is not None:
        slide_mpp = metadata['pixel_size']
        source = "pixel_size (native)"
    elif 'pixel_size_um_embedded' in metadata and metadata['pixel_size_um_embedded'] is not None:
        slide_mpp = metadata['pixel_size_um_embedded']
        source = "pixel_size_um_embedded"
    else:
        raise ValueError(f"No valid pixel size information found in metadata for {wsi_list[i]}")    
        continue
    
    x5_magnification=x5_mpp/slide_mpp
    x20_magnification=x20_mpp/slide_mpp
    x20_patch_image_size=int(patch_image_size*x20_magnification)
    tissue_slide=np.array(slide.get_thumbnail((width//x5_magnification, height//x5_magnification)))
    for row in tqdm(range(height//x20_patch_image_size)):
        for col in range(width//x20_patch_image_size):
            if row*(x20_patch_image_size)<y_min or (row+1)*(x20_patch_image_size)>y_max or col*(x20_patch_image_size)<x_min or (col+1)*(x20_patch_image_size)>x_max:
                continue

            filter_df=annotation_df.loc[(annotation_df['x2']>col*(x20_patch_image_size)) & (annotation_df['x1']<(col+1)*(x20_patch_image_size))]
            filter_df=filter_df.loc[(filter_df['y2']>row*(x20_patch_image_size)) & (filter_df['y1']<(row+1)*(x20_patch_image_size))]
            if filter_df.shape[0]<10:
                continue
            
            patch=slide.read_region(
                (col*(x20_patch_image_size), row*(x20_patch_image_size)),
                0,
                (x20_patch_image_size, x20_patch_image_size)
            ).convert("RGB")
            tissue_patch_x=col*(x20_patch_image_size)-((x20_patch_image_size))//2-((x20_patch_image_size))
            tissue_patch_y=row*(x20_patch_image_size)-((x20_patch_image_size))//2-((x20_patch_image_size))
            if tissue_patch_x<0:
                tissue_patch_x=0
            if tissue_patch_y<0:
                tissue_patch_y=0
            if tissue_patch_x+patch_image_size>width:
                tissue_patch_x=width - patch_image_size
            if tissue_patch_y+patch_image_size>height:
                tissue_patch_y=height - patch_image_size
            tissue_patch=tissue_slide[int(tissue_patch_y//x5_magnification):int(tissue_patch_y//x5_magnification+patch_image_size), int(tissue_patch_x//x5_magnification):int(tissue_patch_x//x5_magnification+patch_image_size), :]
            patch=patch.resize((patch_image_size,patch_image_size))
            pre_df=pd.DataFrame(columns=['x','y','w','h','class'])
            for k in range(len(filter_df)): #x,y,w,h 
                cell_class=filter_df.iloc[k]['class_name']
                y=int((filter_df.iloc[k]['y1']+filter_df.iloc[k]['y2'])//2 - row*(x20_patch_image_size))/(x20_patch_image_size)
                x=int((filter_df.iloc[k]['x1']+filter_df.iloc[k]['x2'])//2 - col*(x20_patch_image_size))/(x20_patch_image_size)
                w=int((filter_df.iloc[k]['x2'] - filter_df.iloc[k]['x1']))/x20_patch_image_size
                h=int((filter_df.iloc[k]['y2'] - filter_df.iloc[k]['y1']))/x20_patch_image_size
                if y>1:
                    y=1
                if x>1:
                    x=1
                if y<0:
                    y=0
                if x<0:
                    x=0    
                pre_df.loc[len(pre_df)] = {'x':x, 'y':y, 'w':w, 'h':h, 'class':class_list_inv[cell_class]}
            save_image_dir=f'../../data/spatialTranscriptome/preprocessed_xenium/patch_train_data/{os.path.basename(wsi_list[i]).replace(".tif","")}/image/'
            save_annotation_dir=f'../../data/spatialTranscriptome/preprocessed_xenium/patch_train_data/{os.path.basename(wsi_list[i]).replace(".tif","")}/annotation/'
            save_tissue_dir=f'../../data/spatialTranscriptome/preprocessed_xenium/patch_train_data/{os.path.basename(wsi_list[i]).replace(".tif","")}/tissue_image/'
            create_dir(save_image_dir)
            create_dir(save_annotation_dir)
            create_dir(save_tissue_dir)
            patch.save(f'{save_image_dir}patch_{row*x20_patch_image_size}_{col*x20_patch_image_size}.png')
            Image.fromarray(tissue_patch).save(f'{save_tissue_dir}patch_{row*x20_patch_image_size}_{col*x20_patch_image_size}.png')
            pre_df.to_csv(f'{save_annotation_dir}patch_{row*x20_patch_image_size}_{col*x20_patch_image_size}.csv', index=False)

In [ ]:
len(wsi_list)

In [ ]:
slide = ops.OpenSlide(wsi_list[7])

# MPP 정보 확인
if 'openslide.mpp-x' in slide.properties:
    mpp_x = float(slide.properties['openslide.mpp-x'])
    print(f"MPP-X: {mpp_x}")

if 'openslide.mpp-y' in slide.properties:
    mpp_y = float(slide.properties['openslide.mpp-y'])
    print(f"MPP-Y: {mpp_y}")

# 평균 MPP 사용
if 'openslide.mpp-x' in slide.properties and 'openslide.mpp-y' in slide.properties:
    slide_mpp = (float(slide.properties['openslide.mpp-x']) + float(slide.properties['openslide.mpp-y'])) / 2

In [ ]:
wsi_list[7]